In [7]:
import pandas as pd
import numpy as np
import joblib
import os
import optuna

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import classification_report, confusion_matrix, f1_score, recall_score, roc_auc_score
from xgboost import XGBClassifier

print("Dependencies loaded successfully.")

Dependencies loaded successfully.


In [8]:

# 1. Load the CSV natively (Pandas will automatically use your header row)
df = pd.read_csv('heart.csv', na_values='?')

# 2. Rename the 'condition' column to 'target' to match the pipeline specs
if 'condition' in df.columns:
    df = df.rename(columns={'condition': 'target'})

# 3. Drop any rows where the target might be NaN just to be safe
df = df.dropna(subset=['target'])

# 4. Ensure target is numeric, then apply binary logic
df['target'] = pd.to_numeric(df['target'], errors='coerce')
df['target'] = df['target'].apply(lambda x: 1 if x > 0 else 0)

print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(df['target'].value_counts())

Dataset loaded: 297 rows, 14 columns
target
0    160
1    137
Name: count, dtype: int64


In [9]:
numeric_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
categorical_features = ['cp', 'restecg', 'slope', 'thal']
# We pass binary/ordinal features through untouched (they don't need scaling or OHE)
passthrough_features = ['sex', 'fbs', 'exang', 'ca']

# 1. Numeric Pipeline: Impute missing with median -> Standardize
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 2. Categorical Pipeline: Impute missing with most frequent -> One-Hot Encode
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    # handle_unknown='ignore' prevents crashes if unseen categories pop up in production
    ('onehot', OneHotEncoder(handle_unknown='ignore')) 
])

# 3. Passthrough Pipeline: Just impute missing values
passthrough_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

# Combine everything into a ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
        ('pass', passthrough_transformer, passthrough_features)
    ])

In [10]:
X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Fit the preprocessor on training data and transform both sets
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Preprocessing pipeline built and data transformed.")


Preprocessing pipeline built and data transformed.


In [11]:
def objective(trial):
    # Define the search space for XGBoost
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 5),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 1.5) # Helps with slight imbalance
    }
    
    clf = XGBClassifier(**params, random_state=42, use_label_encoder=False, eval_metric='logloss')
    
    # We optimize for Recall as per your clinical documentation (catch false negatives)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(clf, X_train_processed, y_train, cv=cv, scoring='recall')
    
    return scores.mean()

print("Spinning up Optuna for hyperparameter tuning...")
study = optuna.create_study(direction='maximize')
# Using 50 trials as per your documentation
study.optimize(objective, n_trials=50)

print("\nBest hyperparameters found by Optuna:")
print(study.best_params)

[I 2026-05-13 16:26:15,323] A new study created in memory with name: no-name-ff19f95e-237f-4837-8536-4bc6c2467acc


Spinning up Optuna for hyperparameter tuning...


c:\Users\Vishn\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\xgboost\training.py:200: UserWarning: [16:26:17] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Vishn\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\xgboost\training.py:200: UserWarning: [16:26:17] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Vishn\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\xgboost\training.py:200: UserWarning: [16:26:17] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Vishn\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\xgboost\training.py:200: UserWarning: [16:26:17] WARNING: C:\actio


Best hyperparameters found by Optuna:
{'n_estimators': 57, 'max_depth': 9, 'learning_rate': 0.012530953041394859, 'subsample': 0.6669747136351893, 'colsample_bytree': 0.9807151496010075, 'min_child_weight': 5, 'scale_pos_weight': 1.3390051537779248}


In [12]:
best_params = study.best_params
final_model = XGBClassifier(**best_params, random_state=42, use_label_encoder=False, eval_metric='logloss')

final_model.fit(X_train_processed, y_train)
y_pred = final_model.predict(X_test_processed)
y_prob = final_model.predict_proba(X_test_processed)[:, 1]

print("\n--- Final Model Evaluation ---")
print(classification_report(y_test, y_pred))
print(f"Recall: {recall_score(y_test, y_pred):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred):.4f}")
print(f"AUC-ROC: {roc_auc_score(y_test, y_prob):.4f}")

c:\Users\Vishn\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\xgboost\training.py:200: UserWarning: [16:26:25] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



--- Final Model Evaluation ---
              precision    recall  f1-score   support

           0       0.88      0.91      0.89        32
           1       0.89      0.86      0.87        28

    accuracy                           0.88        60
   macro avg       0.88      0.88      0.88        60
weighted avg       0.88      0.88      0.88        60

Recall: 0.8571
F1-Score: 0.8727
AUC-ROC: 0.9353


In [13]:
ARTIFACT_DIR = '../backend/model/artifacts'
os.makedirs(ARTIFACT_DIR, exist_ok=True)

MODEL_PATH = os.path.join(ARTIFACT_DIR, 'xgboost_model.joblib')
PREPROCESSOR_PATH = os.path.join(ARTIFACT_DIR, 'preprocessor.joblib')

joblib.dump(final_model, MODEL_PATH)
joblib.dump(preprocessor, PREPROCESSOR_PATH)

print(f"\nPipeline complete! Artifacts successfully saved to: {ARTIFACT_DIR}")
print("Your FastAPI server is now ready to serve predictions.")


Pipeline complete! Artifacts successfully saved to: ../backend/model/artifacts
Your FastAPI server is now ready to serve predictions.
